# An audit built with your agent

Codex used the ITCA+ skill to design and run this workflow. It answers one question
with the package's 560-cell teaching dataset: **which group distinctions are
predictable from molecular features, and which from QC?** No download is required.

## The task

> Use $gauge to audit the grouping `group` in the teaching example.
> Inspect its inputs, choose appropriate views and settings, and explain your
> choices. Show a table and a figure that help me understand the group contrasts.

The request supplies a goal rather than classifier, fold count or screening parameters.

In [ ]:
import json
from pathlib import Path
import tempfile

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import pandas as pd

import gauge_sc as gauge

adata = gauge.datasets.audit_example()
before = adata.copy()
support = adata.obs["group"].value_counts().sort_index()
display(pd.DataFrame({"group": support.index, "cells": support.values}))
print(adata.uns["gauge_sc_example"]["expression_scale"])
print(adata.uns["gauge_sc_example"]["qc_provenance"])
print("Supplied molecular representation:", adata.obsm["X_pca"].shape)

## The agent's decisions for this dataset

| Choice | Reason |
|---|---|
| Use the supplied `X_pca` as the molecular view | The example supplies a fixed 15-component representation; no new preprocessing is needed. |
| Use the three numeric QC columns; log1p the two count-like columns | Counts span a wide range, while the mitochondrial percentage stays on its supplied scale. `simulation_role` is a design annotation and is excluded from features. |
| Five paired folds, pooled recall, seed 0 | Every group has 80 cells, yielding 16 held-out cells per group per fold. The run is small enough to use the default random forest. |
| At least 10 test cells per group/fold; screening disabled | Support is sufficient. The question asks for contrasts, so continuous scores answer it without a binary candidate rule. |

These are choices for this run, not mandatory skill parameters. Numeric QC
standardization happens within training folds.

In [ ]:
views = {
    "molecular": {"obsm": "X_pca"},
    "qc": {
        "obs": ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
        "log1p": ["total_counts", "n_genes_by_counts"],
    },
}
evaluation = gauge.evaluate(
    adata, groupby="group", views=views,
    n_splits=5, random_state=0, aggregation="pooled",
)
audit = gauge.audit(
    evaluation, reference="molecular", contrast="qc", min_test_cells=10,
)
display(evaluation.table[["view", "n_cells", "n_groups", "itca"]].round(3))

In [ ]:
readout = audit.table.sort_values("contrast", ascending=False, kind="stable")
display(readout[["group", "reference_contribution", "contrast_contribution", "contrast"]].round(3))
audit_figure = gauge.pl.audit(
    audit, annotate="all", fontfamily="Inter", context="paper", show=False,
)
plt.show()

## The agent's interpretation

Read the table and view-specific recalls together. The report below is generated from this run's results, rather than a fixed list of expected groups.

In [ ]:
def recall_range(values):
    low, high = values.min(), values.max()
    return f"{low:.3f}" if low == high else f"{low:.3f}–{high:.3f}"

molecular_supported = readout.loc[readout.reference_recall > readout.contrast_recall]
qc_supported = readout.loc[readout.contrast_recall > readout.reference_recall]
molecular_names = ", ".join(sorted(molecular_supported.group))
qc_names = ", ".join(sorted(qc_supported.group))
scores = evaluation.table.set_index("view").itca
agent_interpretation = (
    f"**The interpretation:** {molecular_names} have higher molecular support "
    f"(molecular recall {recall_range(molecular_supported.reference_recall)}). "
    f"{qc_names} have higher QC support "
    f"(QC recall {recall_range(qc_supported.contrast_recall)}, "
    f"molecular recall {recall_range(qc_supported.reference_recall)}).\n\n"
    f"The global molecular ITCA ({scores['molecular']:.3f}) exceeds QC ITCA "
    f"({scores['qc']:.3f}), but the group-level readout reveals which distinctions "
    "rely more on QC. An audit adds information beyond ranking whole groupings. "
    "Here the positive centered contrasts also have positive raw "
    "QC-minus-molecular differences."
)
display(Markdown(agent_interpretation))

## Keep the result available for follow-up

Save the evaluation, then load it for another audit without fitting the classifier
again. This example uses a temporary directory so rerunning the notebook is safe;
your agent can use a new persistent output directory for your own analysis.

In [ ]:
with tempfile.TemporaryDirectory(prefix="gauge_sc-agent-example-") as directory:
    evaluation.save(Path(directory) / "evaluation")
    restored = gauge.io.load_result(Path(directory) / "evaluation")
    resumed = gauge.audit(
        restored, reference="molecular", contrast="qc", min_test_cells=10,
    )
    pd.testing.assert_frame_equal(audit.table, resumed.table, check_exact=True)

pd.testing.assert_frame_equal(before.obs, adata.obs)
assert (before.X != adata.X).nnz == 0
assert evaluation.groups.min_test_cells.min() == 16
assert audit.table.eligible.all()
assert "candidate" not in audit.table
print("Saved-result audit matches exactly; inputs are unchanged.")

## Work with your agent

[Install the skill](../docs/agents.md), then adapt the task above to your data.
Explain what your grouping means and which scientific distinction matters.
If you want screening calls or a different selection scale, ask the agent to
recommend settings and explain their consequences.

Continue with [the real human–mouse mixture](real_data/11_mixture_audit.ipynb)
or [branch-specific spatial selection](real_data/12_spatial_hierarchy.ipynb).